# Distributionally robust promotion planning with demand displacement

This notebook compares three pricing technologies:

1. a myopic promotion rule;
2. a nominal dynamic optimizer;
3. a fixed-radius Wasserstein-DRO dynamic optimizer.

The revised behavioral model separates the ordinary price response,
the contemporaneous promotion lift, and future demand displacement:

\[
Q_t
=
Q_0(1-d_t)^{-\varepsilon}
\exp\!\left(
    \gamma\mathbf 1\{d_t>0\}
    -\psi I_t
\right),
\]

\[
I_{t+1}=rI_t+d_t.
\]

The state \(I_t\) is an aggregate promotion-induced demand-
displacement stock, not a directly observed household inventory.
Promotions are modeled as isolated one-week interventions, consistent
with the empirical calibration, by imposing a minimum number of
non-promotion weeks after each promotion.

The nominal policy maximizes expected discounted profit under the
weighted behavioral distribution from notebook 04. The DRO policy
solves a rectangular robust Bellman recursion over a Wasserstein ball
around that distribution.

The fixed radius is calibrated from resampled distribution instability
and is accompanied by an explicit sensitivity path. The main
contribution is the economic comparison between myopic, nominal
dynamic, and robust pricing technologies—not adaptive radius design.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import (
    linear_sum_assignment,
    minimize_scalar,
)
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

CURRENT_DIR = Path.cwd().resolve()
POSSIBLE_ROOTS = [
    CURRENT_DIR,
    CURRENT_DIR.parent,
]

PROJECT_ROOT = next(
    (
        root
        for root in POSSIBLE_ROOTS
        if (
            root
            / "data"
            / "processed"
        ).is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root "
        f"from {CURRENT_DIR}"
    )

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)
TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
)
FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)
MODEL_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
)

for directory in [
    PROCESSED_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    MODEL_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

PARAMETER_PATH = (
    PROCESSED_DIR
    / "stockpiling_behavioral_draws.pkl"
)

RANDOM_SEED = 42
HORIZON = 12
DISCOUNT_FACTOR = 0.995

DISCOUNT_ACTIONS = np.array(
    [
        0.00,
        0.10,
        0.20,
        0.30,
    ],
    dtype=float,
)

# A promotion must be followed by this many non-promotion weeks.
MIN_NONPROMOTION_WEEKS_AFTER_PROMOTION = 2
COOLDOWN_STATES = np.arange(
    MIN_NONPROMOTION_WEEKS_AFTER_PROMOTION + 1,
    dtype=int,
)

INVENTORY_GRID_SIZE = 61
INITIAL_INVENTORY = 0.0
INITIAL_COOLDOWN = 0
INVENTORY_GRID_BUFFER = 1.20

MAX_DRO_SCENARIOS = 40

# Fixed-radius calibration and sensitivity.
RADIUS_RESAMPLE_REPLICATIONS = 100
RADIUS_RESAMPLE_SIZE = 100
RADIUS_QUANTILE = 0.90
RADIUS_MULTIPLIERS = np.array(
    [
        0.0,
        0.5,
        1.0,
        1.5,
        2.0,
    ],
    dtype=float,
)

PARAMETER_COLUMNS = [
    "price_elasticity",
    "promotion_lift_log",
    "displacement_strength",
    "inventory_persistence",
]

POLICY_OUTPUT_PATH = (
    PROCESSED_DIR
    / "dro_promotion_policies_revised.pkl"
)
EVALUATION_OUTPUT_PATH = (
    PROCESSED_DIR
    / "dro_promotion_evaluation_revised.pkl"
)
SUMMARY_TABLE_PATH = (
    TABLE_DIR
    / "dro_promotion_policy_summary_revised.csv"
)
GROUP_TABLE_PATH = (
    TABLE_DIR
    / "dro_promotion_group_summary_revised.csv"
)
SCENARIO_TABLE_PATH = (
    TABLE_DIR
    / "dro_behavioral_scenarios_revised.csv"
)
RADIUS_TABLE_PATH = (
    TABLE_DIR
    / "dro_radius_calibration_revised.csv"
)
FRONTIER_TABLE_PATH = (
    TABLE_DIR
    / "dro_profit_regret_frontier_revised.csv"
)
FRONTIER_FIGURE_PATH = (
    FIGURE_DIR
    / "dro_profit_regret_frontier_revised.png"
)
PATH_FIGURE_PATH = (
    FIGURE_DIR
    / "dro_representative_policy_path_revised.png"
)
CONFIGURATION_PATH = (
    MODEL_DIR
    / "dro_promotion_planning_revised_configuration.json"
)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Parameter artifact:", PARAMETER_PATH)

## 2. Load and validate behavioral draws

In [ ]:
if not PARAMETER_PATH.is_file():
    raise FileNotFoundError(
        "Run "
        "04_stockpiling_calibration_revised.ipynb "
        "first."
    )

parameter_draws = pd.read_pickle(
    PARAMETER_PATH
).copy()

required_columns = set(
    PARAMETER_COLUMNS
).union(
    {
        "base_demand",
        "regular_price",
        "unit_cost",
        "draw_weight",
    }
)

missing = required_columns.difference(
    parameter_draws.columns
)

if missing:
    raise ValueError(
        "Parameter artifact is missing columns: "
        f"{sorted(missing)}"
    )

for column in required_columns:
    parameter_draws[column] = (
        pd.to_numeric(
            parameter_draws[
                column
            ],
            errors="coerce",
        )
    )

valid = np.ones(
    len(parameter_draws),
    dtype=bool,
)

for column in required_columns:
    valid &= (
        parameter_draws[
            column
        ].notna()
        & np.isfinite(
            parameter_draws[
                column
            ]
        )
    )

valid &= parameter_draws[
    "draw_weight"
].gt(0)

parameter_draws = (
    parameter_draws.loc[
        valid
    ]
    .reset_index(drop=True)
)

if len(parameter_draws) < 20:
    raise RuntimeError(
        "At least 20 valid behavioral draws "
        "are required."
    )

parameter_draws[
    "draw_weight"
] = (
    parameter_draws[
        "draw_weight"
    ]
    / parameter_draws[
        "draw_weight"
    ].sum()
)

BASE_DEMAND = float(
    np.average(
        parameter_draws[
            "base_demand"
        ],
        weights=parameter_draws[
            "draw_weight"
        ],
    )
)
REGULAR_PRICE = float(
    np.average(
        parameter_draws[
            "regular_price"
        ],
        weights=parameter_draws[
            "draw_weight"
        ],
    )
)
UNIT_COST = float(
    np.average(
        parameter_draws[
            "unit_cost"
        ],
        weights=parameter_draws[
            "draw_weight"
        ],
    )
)

if UNIT_COST >= REGULAR_PRICE:
    warnings.warn(
        "The calibrated unit cost is not below "
        "the regular price. Promotion "
        "profitability may be degenerate."
    )

print(
    "Behavioral draws:",
    len(parameter_draws),
)
print(
    "Total probability mass:",
    f"{parameter_draws['draw_weight'].sum():.6f}",
)
print("Base demand:", BASE_DEMAND)
print("Regular price:", REGULAR_PRICE)
print("Unit cost:", UNIT_COST)

display(
    parameter_draws[
        PARAMETER_COLUMNS
    ].describe(
        percentiles=[
            0.05,
            0.50,
            0.95,
        ]
    ).T
)

## 3. Reduce the empirical distribution to weighted medoids

In [ ]:
theta = parameter_draws[
    PARAMETER_COLUMNS
].to_numpy(dtype=float)

original_weights = parameter_draws[
    "draw_weight"
].to_numpy(dtype=float)

theta_mean = np.average(
    theta,
    axis=0,
    weights=original_weights,
)

centered = (
    theta - theta_mean
)

theta_variance = np.average(
    centered ** 2,
    axis=0,
    weights=original_weights,
)

theta_scale = np.sqrt(
    theta_variance
)
theta_scale = np.where(
    theta_scale > 1e-10,
    theta_scale,
    1.0,
)

theta_standardized = (
    theta - theta_mean
) / theta_scale

scenario_count = min(
    MAX_DRO_SCENARIOS,
    len(parameter_draws),
)

if scenario_count == len(
    parameter_draws
):
    medoid_indices = np.arange(
        len(parameter_draws)
    )
    scenario_weights = (
        original_weights.copy()
    )
else:
    kmeans = KMeans(
        n_clusters=scenario_count,
        random_state=RANDOM_SEED,
        n_init=20,
    )

    kmeans.fit(
        theta_standardized,
        sample_weight=original_weights,
    )

    labels = kmeans.labels_
    medoid_indices = []
    scenario_weights = []

    for cluster_id in range(
        scenario_count
    ):
        members = np.flatnonzero(
            labels == cluster_id
        )

        center = (
            kmeans.cluster_centers_[
                cluster_id
            ]
        )

        distances = np.linalg.norm(
            theta_standardized[
                members
            ]
            - center,
            axis=1,
        )

        member_weights = (
            original_weights[
                members
            ]
        )

        weighted_distance = (
            distances
            / np.sqrt(
                member_weights.clip(
                    min=1e-12
                )
            )
        )

        medoid_indices.append(
            int(
                members[
                    np.argmin(
                        weighted_distance
                    )
                ]
            )
        )
        scenario_weights.append(
            float(
                member_weights.sum()
            )
        )

    medoid_indices = np.asarray(
        medoid_indices,
        dtype=int,
    )
    scenario_weights = np.asarray(
        scenario_weights,
        dtype=float,
    )

scenario_weights = (
    scenario_weights
    / scenario_weights.sum()
)

scenarios = parameter_draws.iloc[
    medoid_indices
][
    PARAMETER_COLUMNS
].reset_index(drop=True)

scenarios[
    "scenario_weight"
] = scenario_weights
scenarios[
    "scenario_id"
] = np.arange(
    len(scenarios),
    dtype=int,
)

scenario_theta = scenarios[
    PARAMETER_COLUMNS
].to_numpy(dtype=float)

scenario_standardized = (
    scenario_theta - theta_mean
) / theta_scale

transport_cost = cdist(
    scenario_standardized,
    scenario_standardized,
    metric="euclidean",
)

if not np.isclose(
    scenario_weights.sum(),
    1.0,
):
    raise RuntimeError(
        "Scenario weights do not sum to one."
    )

print(
    "Reduced scenarios:",
    len(scenarios),
)
print(
    "Effective scenario count:",
    f"{1.0 / np.sum(scenario_weights ** 2):.2f}",
)
display(scenarios.head())

## 4. Calibrate a fixed Wasserstein radius

In [ ]:
def equal_weight_empirical_wasserstein(
    sample_a: np.ndarray,
    sample_b: np.ndarray,
) -> float:
    if len(sample_a) != len(sample_b):
        raise ValueError(
            "Samples must have equal cardinality."
        )

    assignment_cost = cdist(
        sample_a,
        sample_b,
        metric="euclidean",
    )

    rows, columns = (
        linear_sum_assignment(
            assignment_cost
        )
    )

    return float(
        assignment_cost[
            rows,
            columns,
        ].mean()
    )


rng = np.random.default_rng(
    RANDOM_SEED
)

resample_size = min(
    RADIUS_RESAMPLE_SIZE,
    max(
        20,
        len(parameter_draws),
    ),
)

radius_rows = []

for replication in range(
    RADIUS_RESAMPLE_REPLICATIONS
):
    first_indices = rng.choice(
        len(theta_standardized),
        size=resample_size,
        replace=True,
        p=original_weights,
    )

    second_indices = rng.choice(
        len(theta_standardized),
        size=resample_size,
        replace=True,
        p=original_weights,
    )

    distance = (
        equal_weight_empirical_wasserstein(
            theta_standardized[
                first_indices
            ],
            theta_standardized[
                second_indices
            ],
        )
    )

    radius_rows.append(
        {
            "replication": replication,
            "wasserstein_distance": distance,
        }
    )

radius_calibration = pd.DataFrame(
    radius_rows
)

CALIBRATED_RHO = float(
    radius_calibration[
        "wasserstein_distance"
    ].quantile(
        RADIUS_QUANTILE
    )
)

RHO_VALUES = np.unique(
    np.round(
        CALIBRATED_RHO
        * RADIUS_MULTIPLIERS,
        8,
    )
)

print(
    "Calibrated rho:",
    f"{CALIBRATED_RHO:.4f}",
)
print(
    "Sensitivity radii:",
    RHO_VALUES,
)
display(
    radius_calibration[
        "wasserstein_distance"
    ].describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
        ]
    )
)
print(
    "Radius interpretation: resampling "
    "instability of the weighted behavioral "
    "distribution, not a formal coverage radius."
)

## 5. Define demand displacement, cooldown, and profit transitions

In [ ]:
@dataclass(frozen=True)
class BehavioralParameters:
    price_elasticity: float
    promotion_lift_log: float
    displacement_strength: float
    inventory_persistence: float


def demand_transition_profit(
    inventory: float,
    cooldown: int,
    discount_depth: float,
    parameters: BehavioralParameters,
) -> tuple[
    float,
    float,
    int,
    float,
]:
    if (
        cooldown > 0
        and discount_depth > 1e-12
    ):
        raise ValueError(
            "A promotion was selected while "
            "the cooldown constraint was active."
        )

    price = (
        REGULAR_PRICE
        * (
            1.0 - discount_depth
        )
    )

    promotion_active = float(
        discount_depth > 1e-12
    )

    price_multiplier = np.power(
        max(
            1.0 - discount_depth,
            1e-6,
        ),
        -parameters.price_elasticity,
    )

    demand = (
        BASE_DEMAND
        * price_multiplier
        * np.exp(
            parameters.promotion_lift_log
            * promotion_active
            - parameters.displacement_strength
            * max(inventory, 0.0)
        )
    )

    next_inventory = (
        parameters.inventory_persistence
        * max(inventory, 0.0)
        + discount_depth
    )

    if discount_depth > 1e-12:
        next_cooldown = (
            MIN_NONPROMOTION_WEEKS_AFTER_PROMOTION
        )
    else:
        next_cooldown = max(
            int(cooldown) - 1,
            0,
        )

    profit = (
        price - UNIT_COST
    ) * demand

    return (
        float(demand),
        float(next_inventory),
        int(next_cooldown),
        float(profit),
    )


maximum_persistence = float(
    parameter_draws[
        "inventory_persistence"
    ].max()
)

INVENTORY_MAX = (
    INVENTORY_GRID_BUFFER
    * float(
        DISCOUNT_ACTIONS.max()
    )
    / max(
        1.0 - maximum_persistence,
        1e-3,
    )
)

INVENTORY_GRID = np.linspace(
    0.0,
    INVENTORY_MAX,
    INVENTORY_GRID_SIZE,
)

scenario_parameters = [
    BehavioralParameters(
        **{
            column: float(
                row[column]
            )
            for column in PARAMETER_COLUMNS
        }
    )
    for _, row in scenarios.iterrows()
]

evaluation_parameters = [
    BehavioralParameters(
        **{
            column: float(
                row[column]
            )
            for column in PARAMETER_COLUMNS
        }
    )
    for _, row in (
        parameter_draws.iterrows()
    )
]

evaluation_weights = (
    parameter_draws[
        "draw_weight"
    ].to_numpy(dtype=float)
)

print(
    "Inventory maximum:",
    f"{INVENTORY_MAX:.3f}",
)
print(
    "Inventory grid points:",
    len(INVENTORY_GRID),
)
print(
    "Cooldown states:",
    COOLDOWN_STATES,
)
print(
    "Discount actions:",
    DISCOUNT_ACTIONS,
)

## 6. Wasserstein worst-case expectation

For scenario payoff vector \(z\), nominal weights \(w\), and
transport-cost matrix \(C\),

\[
\inf_{q:W_C(q,w)\leq\rho}q^\top z
=
\sup_{\lambda\geq0}
\left[
-\lambda\rho+
\sum_iw_i\min_j(z_j+\lambda C_{ij})
\right].
\]

This finite-support dual reduces every inner DRO problem to a
one-dimensional concave optimization.

In [ ]:
def wasserstein_worst_case_expectation(
    scenario_values: np.ndarray,
    nominal_weights: np.ndarray,
    cost_matrix: np.ndarray,
    rho: float,
) -> float:
    values = np.asarray(
        scenario_values,
        dtype=float,
    )
    weights = np.asarray(
        nominal_weights,
        dtype=float,
    )

    if len(values) != len(weights):
        raise ValueError(
            "Scenario values and weights differ in length."
        )

    nominal_value = float(
        np.dot(weights, values)
    )

    if rho <= 1e-12:
        return nominal_value

    value_range = float(
        values.max() - values.min()
    )
    positive_costs = cost_matrix[
        cost_matrix > 1e-12
    ]

    if (
        len(positive_costs) == 0
        or value_range <= 1e-12
    ):
        return float(values.min())

    minimum_positive_cost = float(
        positive_costs.min()
    )
    lambda_upper = (
        value_range
        / minimum_positive_cost
        + 1.0
    )

    def dual_value(lam: float) -> float:
        transported = (
            values[None, :]
            + float(lam) * cost_matrix
        )
        row_minima = transported.min(axis=1)
        return float(
            -float(lam) * rho
            + np.dot(weights, row_minima)
        )

    result = minimize_scalar(
        lambda lam: -dual_value(float(lam)),
        bounds=(0.0, lambda_upper),
        method="bounded",
        options={"xatol": 1e-8},
    )

    robust_value = max(
        dual_value(0.0),
        dual_value(float(result.x)),
        dual_value(lambda_upper),
    )

    return float(
        np.clip(
            robust_value,
            float(values.min()),
            nominal_value,
        )
    )


test_values = np.linspace(
    0.0,
    1.0,
    len(scenarios),
)
nominal_test = (
    wasserstein_worst_case_expectation(
        test_values,
        scenario_weights,
        transport_cost,
        rho=0.0,
    )
)
robust_test = (
    wasserstein_worst_case_expectation(
        test_values,
        scenario_weights,
        transport_cost,
        rho=CALIBRATED_RHO,
    )
)

print("Nominal test value:", nominal_test)
print("Robust test value:", robust_test)

if robust_test > nominal_test + 1e-8:
    raise RuntimeError(
        "Worst-case expectation exceeds nominal expectation."
    )

## 7. Solve myopic, nominal, and DRO feedback policies

In [ ]:
def feasible_actions(
    cooldown: int,
) -> np.ndarray:
    if cooldown > 0:
        return np.array(
            [0.0],
            dtype=float,
        )
    return DISCOUNT_ACTIONS


def scenario_values_for_action(
    inventory: float,
    cooldown: int,
    discount_depth: float,
    continuation_value: np.ndarray | None,
) -> np.ndarray:
    values = []

    for parameters in (
        scenario_parameters
    ):
        (
            _,
            next_inventory,
            next_cooldown,
            profit,
        ) = demand_transition_profit(
            inventory,
            cooldown,
            discount_depth,
            parameters,
        )

        if continuation_value is None:
            total = profit
        else:
            continuation = float(
                np.interp(
                    next_inventory,
                    INVENTORY_GRID,
                    continuation_value[
                        :,
                        next_cooldown,
                    ],
                )
            )
            total = (
                profit
                + DISCOUNT_FACTOR
                * continuation
            )

        values.append(total)

    return np.asarray(
        values,
        dtype=float,
    )


def solve_dynamic_policy(
    rho: float,
) -> dict[str, np.ndarray]:
    value = np.zeros(
        (
            HORIZON + 1,
            INVENTORY_GRID_SIZE,
            len(COOLDOWN_STATES),
        ),
        dtype=float,
    )

    policy = np.zeros(
        (
            HORIZON,
            INVENTORY_GRID_SIZE,
            len(COOLDOWN_STATES),
        ),
        dtype=float,
    )

    action_values = np.full(
        (
            HORIZON,
            INVENTORY_GRID_SIZE,
            len(COOLDOWN_STATES),
            len(DISCOUNT_ACTIONS),
        ),
        np.nan,
        dtype=float,
    )

    for time_index in range(
        HORIZON - 1,
        -1,
        -1,
    ):
        for state_index, inventory in enumerate(
            INVENTORY_GRID
        ):
            for cooldown in COOLDOWN_STATES:
                values = []
                actions = feasible_actions(
                    int(cooldown)
                )

                for discount_depth in actions:
                    scenario_values = (
                        scenario_values_for_action(
                            inventory=float(
                                inventory
                            ),
                            cooldown=int(
                                cooldown
                            ),
                            discount_depth=float(
                                discount_depth
                            ),
                            continuation_value=value[
                                time_index + 1
                            ],
                        )
                    )

                    robust_value = (
                        wasserstein_worst_case_expectation(
                            scenario_values,
                            scenario_weights,
                            transport_cost,
                            rho=float(rho),
                        )
                    )

                    action_index = int(
                        np.flatnonzero(
                            np.isclose(
                                DISCOUNT_ACTIONS,
                                discount_depth,
                            )
                        )[0]
                    )

                    action_values[
                        time_index,
                        state_index,
                        cooldown,
                        action_index,
                    ] = robust_value

                    values.append(
                        (
                            robust_value,
                            float(
                                discount_depth
                            ),
                        )
                    )

                best_value, best_action = max(
                    values,
                    key=lambda item: (
                        item[0],
                        -item[1],
                    ),
                )

                value[
                    time_index,
                    state_index,
                    cooldown,
                ] = best_value

                policy[
                    time_index,
                    state_index,
                    cooldown,
                ] = best_action

    return {
        "rho": float(rho),
        "value": value,
        "policy": policy,
        "action_values": action_values,
    }


def solve_myopic_policy() -> np.ndarray:
    policy = np.zeros(
        (
            HORIZON,
            INVENTORY_GRID_SIZE,
            len(COOLDOWN_STATES),
        ),
        dtype=float,
    )

    for state_index, inventory in enumerate(
        INVENTORY_GRID
    ):
        for cooldown in COOLDOWN_STATES:
            action_profit = []

            for discount_depth in feasible_actions(
                int(cooldown)
            ):
                scenario_values = (
                    scenario_values_for_action(
                        inventory=float(
                            inventory
                        ),
                        cooldown=int(
                            cooldown
                        ),
                        discount_depth=float(
                            discount_depth
                        ),
                        continuation_value=None,
                    )
                )

                action_profit.append(
                    (
                        float(
                            np.dot(
                                scenario_weights,
                                scenario_values,
                            )
                        ),
                        float(
                            discount_depth
                        ),
                    )
                )

            _, best_action = max(
                action_profit,
                key=lambda item: (
                    item[0],
                    -item[1],
                ),
            )

            policy[
                :,
                state_index,
                cooldown,
            ] = best_action

    return policy


dynamic_solutions = {
    float(rho): (
        solve_dynamic_policy(
            float(rho)
        )
    )
    for rho in RHO_VALUES
}

myopic_policy = (
    solve_myopic_policy()
)

no_promotion_policy = np.zeros(
    (
        HORIZON,
        INVENTORY_GRID_SIZE,
        len(COOLDOWN_STATES),
    ),
    dtype=float,
)

policy_dictionary = {
    "no_promotion": (
        no_promotion_policy
    ),
    "myopic": myopic_policy,
    "nominal_dynamic": (
        dynamic_solutions[
            0.0
        ]["policy"]
    ),
}

for rho, solution in (
    dynamic_solutions.items()
):
    if rho == 0.0:
        continue

    policy_dictionary[
        f"dro_rho_{rho:.6f}"
    ] = solution["policy"]

print(
    "Solved policies:",
    list(policy_dictionary),
)

## 8. Solve deterministic oracle policies and evaluate technologies

In [ ]:
def nearest_inventory_index(
    inventory: float,
) -> int:
    return int(
        np.argmin(
            np.abs(
                INVENTORY_GRID
                - inventory
            )
        )
    )


def evaluate_policy(
    policy: np.ndarray,
    parameters: BehavioralParameters,
    initial_inventory: float = (
        INITIAL_INVENTORY
    ),
    initial_cooldown: int = (
        INITIAL_COOLDOWN
    ),
) -> dict[str, object]:
    inventory = float(
        initial_inventory
    )
    cooldown = int(
        initial_cooldown
    )
    total_profit = 0.0

    discount_path = []
    inventory_path = [inventory]
    cooldown_path = [cooldown]
    demand_path = []
    profit_path = []

    for time_index in range(
        HORIZON
    ):
        state_index = (
            nearest_inventory_index(
                inventory
            )
        )

        discount_depth = float(
            policy[
                time_index,
                state_index,
                cooldown,
            ]
        )

        (
            demand,
            next_inventory,
            next_cooldown,
            profit,
        ) = demand_transition_profit(
            inventory,
            cooldown,
            discount_depth,
            parameters,
        )

        total_profit += (
            DISCOUNT_FACTOR
            ** time_index
        ) * profit

        discount_path.append(
            discount_depth
        )
        inventory_path.append(
            next_inventory
        )
        cooldown_path.append(
            next_cooldown
        )
        demand_path.append(
            demand
        )
        profit_path.append(
            profit
        )

        inventory = float(
            np.clip(
                next_inventory,
                0.0,
                INVENTORY_MAX,
            )
        )
        cooldown = int(
            next_cooldown
        )

    positive_discounts = [
        depth
        for depth in discount_path
        if depth > 1e-12
    ]

    return {
        "discounted_profit": float(
            total_profit
        ),
        "promotion_count": int(
            len(positive_discounts)
        ),
        "mean_promotion_depth": (
            float(
                np.mean(
                    positive_discounts
                )
            )
            if positive_discounts
            else 0.0
        ),
        "discount_path": (
            discount_path
        ),
        "inventory_path": (
            inventory_path
        ),
        "cooldown_path": (
            cooldown_path
        ),
        "demand_path": demand_path,
        "profit_path": profit_path,
    }


def solve_oracle_policy(
    parameters: BehavioralParameters,
) -> np.ndarray:
    value = np.zeros(
        (
            HORIZON + 1,
            INVENTORY_GRID_SIZE,
            len(COOLDOWN_STATES),
        ),
        dtype=float,
    )

    policy = np.zeros(
        (
            HORIZON,
            INVENTORY_GRID_SIZE,
            len(COOLDOWN_STATES),
        ),
        dtype=float,
    )

    for time_index in range(
        HORIZON - 1,
        -1,
        -1,
    ):
        for state_index, inventory in enumerate(
            INVENTORY_GRID
        ):
            for cooldown in COOLDOWN_STATES:
                values = []

                for discount_depth in feasible_actions(
                    int(cooldown)
                ):
                    (
                        _,
                        next_inventory,
                        next_cooldown,
                        profit,
                    ) = demand_transition_profit(
                        float(inventory),
                        int(cooldown),
                        float(
                            discount_depth
                        ),
                        parameters,
                    )

                    continuation = float(
                        np.interp(
                            next_inventory,
                            INVENTORY_GRID,
                            value[
                                time_index + 1,
                                :,
                                next_cooldown,
                            ],
                        )
                    )

                    values.append(
                        (
                            profit
                            + DISCOUNT_FACTOR
                            * continuation,
                            float(
                                discount_depth
                            ),
                        )
                    )

                best_value, best_action = max(
                    values,
                    key=lambda item: (
                        item[0],
                        -item[1],
                    ),
                )

                value[
                    time_index,
                    state_index,
                    cooldown,
                ] = best_value

                policy[
                    time_index,
                    state_index,
                    cooldown,
                ] = best_action

    return policy


evaluation_rows = []

for draw_index, parameters in enumerate(
    evaluation_parameters
):
    oracle_policy = (
        solve_oracle_policy(
            parameters
        )
    )

    oracle_result = (
        evaluate_policy(
            oracle_policy,
            parameters,
        )
    )

    oracle_profit = float(
        oracle_result[
            "discounted_profit"
        ]
    )

    for policy_name, policy in (
        policy_dictionary.items()
    ):
        result = evaluate_policy(
            policy,
            parameters,
        )

        evaluation_rows.append(
            {
                "draw_index": (
                    draw_index
                ),
                "draw_weight": float(
                    evaluation_weights[
                        draw_index
                    ]
                ),
                "policy": (
                    policy_name
                ),
                "discounted_profit": (
                    result[
                        "discounted_profit"
                    ]
                ),
                "oracle_profit": (
                    oracle_profit
                ),
                "regret": (
                    oracle_profit
                    - result[
                        "discounted_profit"
                    ]
                ),
                "promotion_count": (
                    result[
                        "promotion_count"
                    ]
                ),
                "mean_promotion_depth": (
                    result[
                        "mean_promotion_depth"
                    ]
                ),
                "price_elasticity": (
                    parameters.price_elasticity
                ),
                "promotion_lift_log": (
                    parameters.promotion_lift_log
                ),
                "displacement_strength": (
                    parameters.displacement_strength
                ),
                "inventory_persistence": (
                    parameters.inventory_persistence
                ),
            }
        )

policy_evaluation = pd.DataFrame(
    evaluation_rows
)

policy_evaluation[
    "displacement_burden"
] = (
    policy_evaluation[
        "displacement_strength"
    ]
    / (
        1.0
        - policy_evaluation[
            "inventory_persistence"
        ]
    ).clip(
        lower=1e-6
    )
)

print(
    "Evaluation rows:",
    len(policy_evaluation),
)

## 9. Technology value and regret summaries

In [ ]:
def weighted_metric(
    frame: pd.DataFrame,
    value_column: str,
) -> float:
    return float(
        np.average(
            frame[
                value_column
            ],
            weights=frame[
                "draw_weight"
            ],
        )
    )


def weighted_quantile(
    values: np.ndarray,
    weights: np.ndarray,
    quantile: float,
) -> float:
    order = np.argsort(values)
    ordered_values = values[
        order
    ]
    ordered_weights = weights[
        order
    ]
    cumulative = np.cumsum(
        ordered_weights
    )
    cumulative = (
        cumulative
        / cumulative[-1]
    )

    return float(
        np.interp(
            quantile,
            cumulative,
            ordered_values,
        )
    )


summary_rows = []

for policy_name, group in (
    policy_evaluation.groupby(
        "policy",
        observed=True,
    )
):
    values = group[
        "discounted_profit"
    ].to_numpy(dtype=float)
    regrets = group[
        "regret"
    ].to_numpy(dtype=float)
    weights = group[
        "draw_weight"
    ].to_numpy(dtype=float)

    summary_rows.append(
        {
            "policy": (
                policy_name
            ),
            "mean_profit": (
                weighted_metric(
                    group,
                    "discounted_profit",
                )
            ),
            "p10_profit": (
                weighted_quantile(
                    values,
                    weights,
                    0.10,
                )
            ),
            "worst_profit": float(
                values.min()
            ),
            "mean_regret": (
                weighted_metric(
                    group,
                    "regret",
                )
            ),
            "p90_regret": (
                weighted_quantile(
                    regrets,
                    weights,
                    0.90,
                )
            ),
            "maximum_regret": float(
                regrets.max()
            ),
            "mean_promotion_count": (
                weighted_metric(
                    group,
                    "promotion_count",
                )
            ),
            "mean_promotion_depth": (
                weighted_metric(
                    group,
                    "mean_promotion_depth",
                )
            ),
        }
    )

policy_summary = pd.DataFrame(
    summary_rows
)

draw_burden = (
    policy_evaluation[
        [
            "draw_index",
            "displacement_burden",
        ]
    ]
    .drop_duplicates(
        "draw_index"
    )
)

draw_burden[
    "displacement_group"
] = pd.qcut(
    draw_burden[
        "displacement_burden"
    ],
    q=3,
    labels=[
        "low",
        "moderate",
        "high",
    ],
    duplicates="drop",
)

evaluation_by_group = (
    policy_evaluation.merge(
        draw_burden[
            [
                "draw_index",
                "displacement_group",
            ]
        ],
        on="draw_index",
        how="left",
        validate="many_to_one",
    )
)

group_rows = []

for (
    displacement_group,
    policy_name,
), group in evaluation_by_group.groupby(
    [
        "displacement_group",
        "policy",
    ],
    observed=True,
):
    group_rows.append(
        {
            "displacement_group": (
                str(displacement_group)
            ),
            "policy": (
                str(policy_name)
            ),
            "mean_profit": (
                weighted_metric(
                    group,
                    "discounted_profit",
                )
            ),
            "mean_regret": (
                weighted_metric(
                    group,
                    "regret",
                )
            ),
            "mean_promotions": (
                weighted_metric(
                    group,
                    "promotion_count",
                )
            ),
            "mean_depth": (
                weighted_metric(
                    group,
                    "mean_promotion_depth",
                )
            ),
        }
    )

group_summary = pd.DataFrame(
    group_rows
)

display(
    policy_summary.sort_values(
        [
            "maximum_regret",
            "mean_regret",
        ]
    )
)
display(group_summary)

## 10. Policy monotonicity diagnostics

In [ ]:
monotonicity_rows = []

for policy_name, policy in (
    policy_dictionary.items()
):
    violations = 0
    comparisons = 0

    for time_index in range(
        HORIZON
    ):
        for cooldown in (
            COOLDOWN_STATES
        ):
            increases = (
                np.diff(
                    policy[
                        time_index,
                        :,
                        cooldown,
                    ]
                )
                > 1e-12
            )

            violations += int(
                increases.sum()
            )
            comparisons += int(
                len(increases)
            )

    monotonicity_rows.append(
        {
            "policy": policy_name,
            "violations": violations,
            "comparisons": comparisons,
            "violation_share": (
                violations
                / comparisons
                if comparisons > 0
                else np.nan
            ),
        }
    )

monotonicity_summary = (
    pd.DataFrame(
        monotonicity_rows
    )
)

display(
    monotonicity_summary
)

print(
    "A violation means the recommended "
    "discount becomes deeper as the "
    "demand-displacement stock increases. "
    "Monotonicity is diagnosed, not imposed."
)

## 11. Profit–regret frontier

In [ ]:
frontier_rows = []

for rho in RHO_VALUES:
    policy_name = (
        "nominal_dynamic"
        if rho == 0.0
        else f"dro_rho_{rho:.6f}"
    )

    row = policy_summary.loc[
        policy_summary[
            "policy"
        ].eq(policy_name)
    ]
    if row.empty:
        continue

    frontier_rows.append(
        {
            "rho": float(rho),
            "mean_profit": float(
                row.iloc[0][
                    "mean_profit"
                ]
            ),
            "p10_profit": float(
                row.iloc[0][
                    "p10_profit"
                ]
            ),
            "maximum_regret": float(
                row.iloc[0][
                    "maximum_regret"
                ]
            ),
            "mean_regret": float(
                row.iloc[0][
                    "mean_regret"
                ]
            ),
            "mean_promotion_count": float(
                row.iloc[0][
                    "mean_promotion_count"
                ]
            ),
            "mean_promotion_depth": float(
                row.iloc[0][
                    "mean_promotion_depth"
                ]
            ),
        }
    )

frontier = pd.DataFrame(
    frontier_rows
).sort_values("rho")

display(frontier)

fig, ax = plt.subplots(
    figsize=(7.5, 4.8)
)
ax.plot(
    frontier[
        "maximum_regret"
    ],
    frontier[
        "mean_profit"
    ],
    marker="o",
)

for row in frontier.itertuples(
    index=False
):
    ax.annotate(
        f"rho={row.rho:.2f}",
        (
            row.maximum_regret,
            row.mean_profit,
        ),
        textcoords="offset points",
        xytext=(5, 5),
    )

ax.set_xlabel(
    "Maximum regret across bootstrap DGPs"
)
ax.set_ylabel(
    "Mean discounted profit"
)
ax.set_title(
    "Profit–robustness frontier"
)
fig.tight_layout()
fig.savefig(
    FRONTIER_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print(
    "Saved:",
    FRONTIER_FIGURE_PATH,
)

## 12. Representative policy paths

In [ ]:
unique_draws = (
    policy_evaluation[
        [
            "draw_index",
            "displacement_burden",
        ]
    ]
    .drop_duplicates(
        "draw_index"
    )
    .sort_values(
        "displacement_burden"
    )
    .reset_index(drop=True)
)

representative_position = int(
    round(
        0.50
        * (
            len(unique_draws) - 1
        )
    )
)

representative_draw_index = int(
    unique_draws.iloc[
        representative_position
    ][
        "draw_index"
    ]
)

representative_parameters = (
    evaluation_parameters[
        representative_draw_index
    ]
)

main_rho = float(
    RHO_VALUES[
        np.argmin(
            np.abs(
                RHO_VALUES
                - CALIBRATED_RHO
            )
        )
    ]
)

main_dro_name = (
    f"dro_rho_{main_rho:.6f}"
)

path_rows = []

for policy_name in [
    "myopic",
    "nominal_dynamic",
    main_dro_name,
]:
    result = evaluate_policy(
        policy_dictionary[
            policy_name
        ],
        representative_parameters,
    )

    for week_index, depth in enumerate(
        result[
            "discount_path"
        ],
        start=1,
    ):
        path_rows.append(
            {
                "policy": (
                    policy_name
                ),
                "week": (
                    week_index
                ),
                "discount_depth": (
                    depth
                ),
                "inventory_before": (
                    result[
                        "inventory_path"
                    ][
                        week_index - 1
                    ]
                ),
                "cooldown_before": (
                    result[
                        "cooldown_path"
                    ][
                        week_index - 1
                    ]
                ),
            }
        )

representative_paths = pd.DataFrame(
    path_rows
)

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

for policy_name, group in (
    representative_paths.groupby(
        "policy",
        observed=True,
    )
):
    ax.step(
        group[
            "week"
        ],
        group[
            "discount_depth"
        ],
        where="mid",
        label=policy_name,
    )

ax.set_xlabel(
    "Planning week"
)
ax.set_ylabel(
    "Discount depth"
)
ax.set_ylim(
    -0.01,
    float(
        DISCOUNT_ACTIONS.max()
    )
    + 0.05,
)
ax.set_title(
    "Promotion paths under a representative "
    "behavioral draw"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    PATH_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

display(
    representative_paths
)
print(
    "Saved:",
    PATH_FIGURE_PATH,
)

## 13. Save outputs

In [ ]:
policy_artifact = {
    "inventory_grid": (
        INVENTORY_GRID
    ),
    "cooldown_states": (
        COOLDOWN_STATES
    ),
    "discount_actions": (
        DISCOUNT_ACTIONS
    ),
    "horizon": HORIZON,
    "discount_factor": (
        DISCOUNT_FACTOR
    ),
    "calibrated_rho": (
        CALIBRATED_RHO
    ),
    "rho_values": RHO_VALUES,
    "policies": (
        policy_dictionary
    ),
    "dynamic_values": {
        float(rho): solution[
            "value"
        ]
        for rho, solution in (
            dynamic_solutions.items()
        )
    },
}

pd.to_pickle(
    policy_artifact,
    POLICY_OUTPUT_PATH,
)

policy_evaluation.to_pickle(
    EVALUATION_OUTPUT_PATH
)

policy_summary.to_csv(
    SUMMARY_TABLE_PATH,
    index=False,
)

group_summary.to_csv(
    GROUP_TABLE_PATH,
    index=False,
)

scenarios.to_csv(
    SCENARIO_TABLE_PATH,
    index=False,
)

radius_calibration.to_csv(
    RADIUS_TABLE_PATH,
    index=False,
)

frontier.to_csv(
    FRONTIER_TABLE_PATH,
    index=False,
)

configuration = {
    "horizon": int(
        HORIZON
    ),
    "discount_factor": float(
        DISCOUNT_FACTOR
    ),
    "discount_actions": (
        DISCOUNT_ACTIONS.tolist()
    ),
    "minimum_nonpromotion_weeks_after_promotion": int(
        MIN_NONPROMOTION_WEEKS_AFTER_PROMOTION
    ),
    "inventory_grid_size": int(
        INVENTORY_GRID_SIZE
    ),
    "inventory_max": float(
        INVENTORY_MAX
    ),
    "dro_scenarios": int(
        len(scenarios)
    ),
    "radius_resample_replications": int(
        RADIUS_RESAMPLE_REPLICATIONS
    ),
    "radius_resample_size": int(
        resample_size
    ),
    "radius_quantile": float(
        RADIUS_QUANTILE
    ),
    "calibrated_rho": float(
        CALIBRATED_RHO
    ),
    "rho_values": (
        RHO_VALUES.tolist()
    ),
    "transport_metric": (
        "euclidean_distance_in_standardized_"
        "behavioral_parameters"
    ),
    "ambiguity_geometry": (
        "finite_support_wasserstein_1_"
        "rectangular_by_period"
    ),
    "behavioral_model": (
        "Q0*(1-d)^(-epsilon)*"
        "exp(gamma*1[d>0]-psi*I);"
        "I_next=r*I+d"
    ),
    "evaluation": (
        "fixed_behavioral_parameters_over_"
        "each_12_week_DGP"
    ),
    "economic_mechanism": (
        "promotion_induced_future_demand_"
        "displacement_separated_from_"
        "contemporaneous_merchandising_lift"
    ),
}

CONFIGURATION_PATH.write_text(
    json.dumps(
        configuration,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved policies:",
    POLICY_OUTPUT_PATH,
)
print(
    "Saved evaluation draws:",
    EVALUATION_OUTPUT_PATH,
)
print(
    "Saved summary:",
    SUMMARY_TABLE_PATH,
)
print(
    "Saved configuration:",
    CONFIGURATION_PATH,
)

## Interpretation checklist

The revised prototype is promising only when:

1. notebook 04 shows a positive contemporaneous promotion lift and a
   positive depth-dependent week-\(+1\) displacement slope;
2. persistence uncertainty is reported honestly when later lag
   slopes do not identify geometric decay;
3. myopic, nominal, and DRO policies do not all choose the same
   action in every feasible state;
4. increasing \(\rho\) improves maximum regret or the lower profit
   tail at a visible cost in mean profit;
5. stronger displacement generally makes deep or frequent
   promotions less attractive;
6. the promotion cooldown prevents the optimizer from repeatedly
   harvesting a one-week merchandising intercept;
7. the maximum discount is not chosen at every available promotion
   opportunity;
8. policy monotonicity violations are investigated rather than
   suppressed;
9. model-based profits are not described as realized causal effects.